In [5]:
import pandas as pd

# Loading the dataset
df = pd.read_csv('multi_touch_attribution_data.csv')

# Inspecting the columns to understand the structure
print("--- Columns ---")
print(df.columns)

# Checking for missing values
print("\n--- Missing Values ---")
print(df.isnull().sum())

# Peeking at the first 5 rows
print("\n--- First 5 Rows ---")
print(df.head())

--- Columns ---
Index(['User ID', 'Timestamp', 'Channel', 'Campaign', 'Conversion'], dtype='str')

--- Missing Values ---
User ID       0
Timestamp     0
Channel       0
Campaign      0
Conversion    0
dtype: int64

--- First 5 Rows ---
   User ID            Timestamp         Channel            Campaign Conversion
0    83281  2025-02-10 07:58:51           Email  New Product Launch         No
1    68071  2025-02-10 23:38:48      Search Ads         Winter Sale         No
2    90131  2025-02-11 10:41:07    Social Media     Brand Awareness        Yes
3    71026  2025-02-10 08:19:44  Direct Traffic                   -        Yes
4    94486  2025-02-10 15:15:46           Email         Retargeting        Yes


In [7]:
# Converting Timestamp to datetime
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

# Sorting by user and time
df = df.sort_values(['User ID', 'Timestamp'])

# Creating the sequence of touchpoints for each user
df['Path'] = df.groupby('User ID')['Channel'].transform(lambda x: ' > '.join(x))

# Getting the final conversion status for each user
user_paths = df.groupby('User ID').agg({
    'Path': 'first',
    'Conversion': 'last'
})

print(user_paths.head())

                                                      Path Conversion
User ID                                                              
10028                             Search Ads > Display Ads        Yes
10045                             Search Ads > Display Ads        Yes
10062                Social Media > Direct Traffic > Email        Yes
10068    Search Ads > Social Media > Social Media > Sea...         No
10095    Display Ads > Email > Referral > Display Ads >...        Yes


In [17]:
import pandas as pd
import numpy as np

# Getting the lists of paths for converted and non-converted users
sequences = [path.split(' > ') for path in user_paths['Path']]

# Extracting all unique channels
all_channels = set([channel for path in sequences for channel in path])
states = sorted(list(all_channels))

# Initializing an empty transition matrix
# We create a square matrix (channels x channels)
matrix = pd.DataFrame(0, index=states, columns=states)

# Filling the matrix by counting transitions
for path in sequences:
    for i in range(len(path) - 1):
        matrix.loc[path[i], path[i+1]] += 1

# Normalizing to probabilities (Row sums = 1)
transition_matrix = matrix.div(matrix.sum(axis=1), axis=0).fillna(0)

print("--- Markov Chain Transition Matrix ---")
print(transition_matrix)

--- Markov Chain Transition Matrix ---
                Direct Traffic  Display Ads     Email  Referral  Search Ads  \
Direct Traffic        0.180165     0.164463  0.166942  0.155372    0.170248   
Display Ads           0.159867     0.183181  0.172356  0.172356    0.159867   
Email                 0.165394     0.161154  0.166243  0.162850    0.179813   
Referral              0.179675     0.173984  0.157724  0.153659    0.156098   
Search Ads            0.178034     0.148804  0.174491  0.180691    0.145261   
Social Media          0.170266     0.154485  0.172757  0.181894    0.151163   

                Social Media  
Direct Traffic      0.162810  
Display Ads         0.152373  
Email               0.164546  
Referral            0.178862  
Search Ads          0.172719  
Social Media        0.169435  


In [19]:
# Defining the removal effect calculation
def calculate_removal_effect(matrix):
    removal_effects = {}
    
    # Baseline conversion rate (sum of the matrix paths)
    baseline_conversion = matrix.values.sum() / len(matrix)
    
    for channel in matrix.index:
        # Creating a copy and zero out the channel
        temp_matrix = matrix.copy()
        temp_matrix.loc[channel, :] = 0
        temp_matrix.loc[:, channel] = 0
        
        # Calculating new conversion rate
        new_conversion = temp_matrix.values.sum() / len(matrix)
        
        # Removal effect = % drop in conversions
        effect = (baseline_conversion - new_conversion) / baseline_conversion
        removal_effects[channel] = effect
        
    return removal_effects

# 2. Running the calculation
effects = calculate_removal_effect(transition_matrix)

# 3. Printing the results
print("--- Removal Effect (Impact of Cutting Budget) ---")
for channel, effect in effects.items():
    print(f"{channel}: {effect:.2%}")

--- Removal Effect (Impact of Cutting Budget) ---
Direct Traffic: 30.89%
Display Ads: 30.05%
Email: 30.74%
Referral: 30.89%
Search Ads: 30.29%
Social Media: 30.52%
